- `async def` creates a **coroutine function**; calling it returns a coroutine object.
- `await` runs a coroutine and returns its result — sequentially by default.
- To get concurrency you must use `gather`, `create_task`, or similar primitives.
- The event loop runs **one thing at a time**, but switches whenever code hits `await`. This is *cooperative* multitasking.

### Why streaming matters for LLM UX

- **Perceived latency drops a lot** — users see the first word in ~100 ms instead of waiting for the full paragraph.
- **You can pipeline downstream work** — e.g. start TTS or display rendering before generation finishes.
- **Cancellation is cheap** — if the user stops reading, you stop consuming tokens and the server can free resources.

### Key takeaways

- `yield` pauses a function and returns a value without ending it.
- Regular generators = lazy sync sequences. Async generators = lazy async sequences.
- LLM streaming is literally an async generator of tokens.


Real systems need more than "run these in parallel". You need to **start tasks in the background, cancel them, time them out, and survive partial failures**.

## The tools

| Primitive                      | What it does                                      |
|--------------------------------|---------------------------------------------------|
| `asyncio.create_task(coro)`    | Schedules a coroutine to run in the background.   |
| `task.cancel()`                | Requests cancellation.                            |
| `asyncio.wait_for(coro, t)`    | Awaits with a timeout; cancels on expiry.         |
| `asyncio.gather(..., return_exceptions=True)` | Keep going even if some tasks raise.  |
| `try/except` inside coroutines | Localize failures so they don't kill siblings.    |


### Key takeaways

- `create_task` = fire-and-remember background work.
- `task.cancel()` raises `CancelledError` inside the task — handle cleanup there.
- `wait_for(coro, timeout=...)` is your friend for bounded latency.
- `gather(..., return_exceptions=True)` to survive partial failure.
- For production: **wrap each coroutine in try/except and return a structured value**. Never let one bad call kill a whole batch.


Four patterns you'll use constantly when building LLM applications:

1. **Streaming chat** — token-by-token output for responsiveness.
2. **Parallel tool execution** — an agent calls N tools at once.
3. **Fan-out / fan-in** — dispatch N prompts, merge the results.
4. **Partial-failure resilience** — one tool fails, the rest keep working.

### Where you'll see this in real LLM systems

- **Agent frameworks** (LangGraph, OpenAI Agents SDK) — every tool call is a coroutine.
- **Multi-model evaluation** — score one answer with 5 judge models in parallel.
- **RAG with reranking** — embed N queries, then rerank top-K, all concurrently.
- **Multi-step pipelines** — generate → critique → revise, with streaming at each step.

### Key takeaways

- Streaming = async generator of tokens.
- Parallel tools = `gather` over independent I/O coroutines.
- Fan-out/fan-in = dispatch many, merge into one.
- Always combine these with the defensive `try/except`-per-task pattern



- LLM apps are I/O-bound → async is a big win.
- `async def` + `await` + event loop = cooperative concurrency on one thread.
- `gather` runs coroutines in parallel and preserves input order.
- `create_task`, `wait_for`, `CancelledError` let you manage lifecycle and timeouts.
- `return_exceptions=True` + per-task `try/except` = resilient batches.
- Generators (`yield`) and async generators (`async def` + `yield`) are the shape of streaming.
- Everyday LLM patterns: streaming, parallel tools, fan-out/fan-in.
- Gradio's streaming UIs are async generators in disguise.


## Cheatsheet

| Task                                   | Snippet                                                          |
|----------------------------------------|------------------------------------------------------------------|
| Define a coroutine                     | `async def f(x): ...`                                            |
| Await a coroutine                      | `result = await f(x)`                                            |
| Run in Jupyter cell                    | `await main()`  &nbsp; *(top-level await)*                       |
| Run from a `.py` script                | `asyncio.run(main())`                                            |
| Run many in parallel                   | `await asyncio.gather(*coros)`                                   |
| Run in background                      | `t = asyncio.create_task(coro)`                                  |
| Cancel a task                          | `t.cancel()` &nbsp; — &nbsp; handle `asyncio.CancelledError`     |
| Timeout                                | `await asyncio.wait_for(coro, timeout=2.0)`                      |
| Survive partial failures               | `gather(..., return_exceptions=True)`                            |
| Streaming producer                     | `async def gen(): ...; yield tok`                                |
| Streaming consumer                     | `async for tok in gen(): ...`                                    |
| Wait for first to finish               | `done, pending = await asyncio.wait(coros, return_when=FIRST_COMPLETED)` |
| Limit concurrency                      | `sem = asyncio.Semaphore(10); async with sem: ...`               |
| Gradio streaming handler               | `async def fn(...): ...; yield ui_update`                        |

## Common pitfalls

- **Forgetting `await`** — you get a coroutine object, not a result. (Python warns you.)
- **Using `time.sleep` in async code** — it blocks the whole event loop. Use `asyncio.sleep`.
- **CPU-heavy work in a coroutine** — also blocks the loop. Offload with `asyncio.to_thread` or a process pool.
- **Unbounded concurrency** — 10,000 parallel requests will crash your rate limit. Use a `Semaphore`.
